In [1]:
import gym_electric_motor as gem
import numpy as np
import scipy.interpolate as sp_interpolate
import gem_controllers as gc
from gym_electric_motor.physical_system_wrappers import FluxObserver





In [ ]:
def _calculate_luts(self):
        """
        Calculates the lookup tables for the maximum torque and the optimal currents for a given torque reference.
        """
        minimum_loss = []
        best_params = []

        minimum_loss_psi = []
        best_params_psi = []

        self.psi_max = self.l_m * self.i_e_lim + self.l_d * self.i_q_lim
        torque = np.linspace(0, self.t_lim, self.t_count)

        self.t_max_psi = np.zeros(self.psi_count)

        for t in torque:
            losses = []
            parameter = []
            for idx, psi in enumerate(np.linspace(0, self.psi_max, self.psi_count)):
                losses_psi = []
                parameter_psi = []
                for i_e in np.linspace(0, self.i_e_lim, self.i_e_count):
                    i_d, i_q = self.solve_analytical(t, psi, i_e)
                    if np.sqrt(i_d**2 + i_q**2) < self.i_q_lim:
                        loss = self.loss(i_d, i_q, i_e)
                        params = np.array([t, psi, i_d, i_q, i_e])
                        losses.append(loss)
                        losses_psi.append(loss)
                        parameter.append(params)
                        parameter_psi.append(params)
                        self.t_max_psi[idx] = t
                if len(losses_psi) > 0:
                    minimum_loss_psi.append(min(losses_psi))
                    best_params_psi.append(parameter_psi[losses_psi.index(minimum_loss_psi[-1])])
            if len(losses) > 0:
                minimum_loss.append(min(losses))
                best_params.append(parameter[losses.index(minimum_loss[-1])])

        best_params = np.array(best_params)
        best_params_psi = np.array(best_params_psi)

        self.t_max_psi = sp_interpolate.interp1d(
            np.linspace(0, self.psi_max, self.psi_count), 0.99 * self.t_max_psi, kind="linear"
        )

        self.t_max = np.max(best_params[:, 0])
        self.psi_opt = sp_interpolate.interp1d(best_params[:, 0], best_params[:, 1], kind="cubic")
        self.i_d_opt = sp_interpolate.interp1d(best_params[:, 0], best_params[:, 2], kind="cubic")
        self.i_q_opt = sp_interpolate.interp1d(best_params[:, 0], best_params[:, 3], kind="cubic")
        self.i_e_opt = sp_interpolate.interp1d(best_params[:, 0], best_params[:, 4], kind="cubic")

        self.t_grid, self.psi_grid = np.mgrid[
            0 : self.t_max : complex(0, self.t_grid_count), 0 : self.psi_max : complex(self.psi_grid_count)
        ]

        self.i_d_inter = sp_interpolate.griddata(
            (best_params_psi[:, 0], best_params_psi[:, 1]),
            best_params_psi[:, 2],
            (self.t_grid, self.psi_grid),
            method="linear",
        )
        self.i_q_inter = sp_interpolate.griddata(
            (best_params_psi[:, 0], best_params_psi[:, 1]),
            best_params_psi[:, 3],
            (self.t_grid, self.psi_grid),
            method="linear",
        )
        self.i_e_inter = sp_interpolate.griddata(
            (best_params_psi[:, 0], best_params_psi[:, 1]),
            best_params_psi[:, 4],
            (self.t_grid, self.psi_grid),
            method="linear",
        )


In [2]:


    # choose the action space
action_space = 'Cont'   # 'Cont' or 'Finite'

    # choose the control task
control_task = 'TC'     # 'SC' (speed control), 'TC' (torque control) or 'CC' (current control)

    # chosse the motor type
motor_type = 'EESM'     # 'PermExDc', 'ExtExDc', 'SeriesDc', 'ShuntDc', 'PMSM', 'EESM', 'SynRM' or 'SCIM'

env_id = action_space + '-' + control_task + '-' + motor_type + '-v0'

    # using a flux observer for the SCIM
physical_system_wrappers = (FluxObserver(),) if motor_type == 'SCIM' else ()

    # Initilize the environment
env = gem.make(env_id, physical_system_wrappers=physical_system_wrappers)
   
    # Initialize the controller
c = gc.GemController.make(
        env,
        env_id,
        a=8,
        block_diagram=False,
        current_safety_margin=0.25,
      #  save_block_diagram_as=(),
    )

    # Control the environment
c.control_environment(env, n_steps=30000, render_env=True, max_episode_length=10000)

AttributeError: module 'numpy' has no attribute 'complex'.
`np.complex` was a deprecated alias for the builtin `complex`. To avoid this error in existing code, use `complex` by itself. Doing this will not modify any behavior and is safe. If you specifically wanted the numpy scalar type, use `np.complex128` here.
The aliases was originally deprecated in NumPy 1.20; for more details and guidance see the original release note at:
    https://numpy.org/devdocs/release/1.20.0-notes.html#deprecations